# Excessive Agency and Tool Permissioning

**Phase 08** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-08/05-excessive-agency-tool-permissioning.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Excessive Agency and Tool Permissioning - Phase 08 Lesson 05
appliedaifromscratch.com

Demonstrates: OWASP LLM06 mitigation via ToolPermissionPolicy.
Enforces per-tool permission levels and human approval gates.

Run:
    python main.py

Requires:
    pip install anthropic
"""

from __future__ import annotations

import json
import datetime
from enum import IntEnum
from dataclasses import dataclass
from typing import Callable

import anthropic

### Permission levels

In [ ]:
class PermissionLevel(IntEnum):
    """
    Ordered permission levels. Higher value = more dangerous.

    READ    - zero side effects, fully reversible (search, query)
    WRITE   - reversible with effort (create, update)
    EXECUTE - hard to reverse (send email, call external API)
    ADMIN   - potentially permanent (delete, drop table, revoke access)
    """
    READ = 1
    WRITE = 2
    EXECUTE = 3
    ADMIN = 4

### Tool specification

In [ ]:
@dataclass
class ToolSpec:
    """One entry in the agent's tool manifest."""
    name: str
    level: PermissionLevel
    description: str
    handler: Callable[..., str]


class PolicyViolation(Exception):
    """Raised when a tool call is blocked by the permission policy."""
    pass

### Permission policy

In [ ]:
class ToolPermissionPolicy:
    """
    Enforces least-privilege tool access for an agent.

    Every tool call goes through this policy before execution.
    The model never runs tools directly.

    Args:
        tools: The tool manifest for this agent.
        max_autonomous_level: Tools at or below this level run without approval.
        gate_fn: Called for tools above max_autonomous_level.
                 Returns True to approve, False to deny.
                 Defaults to stdin prompt.
    """

    def __init__(
        self,
        tools: list[ToolSpec],
        max_autonomous_level: PermissionLevel = PermissionLevel.READ,
        gate_fn: Callable[[str, dict], bool] | None = None,
    ):
        self._tools: dict[str, ToolSpec] = {t.name: t for t in tools}
        self._max_auto = max_autonomous_level
        self._gate_fn = gate_fn or self._default_gate
        self._audit_log: list[dict] = []

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def execute(self, tool_name: str, args: dict) -> str:
        """
        Execute a tool call through the permission policy.

        Returns the tool result string on success.
        Raises PolicyViolation if the call is blocked.
        """
        if tool_name not in self._tools:
            self._log(tool_name, args, "DENIED_UNKNOWN")
            raise PolicyViolation(
                f"Unknown tool: {tool_name!r}. Not in manifest. "
                "Available tools: " + ", ".join(self._tools.keys())
            )

        spec = self._tools[tool_name]

        # ADMIN tools never run autonomously under any circumstance
        if spec.level == PermissionLevel.ADMIN:
            self._log(tool_name, args, "DENIED_ADMIN")
            raise PolicyViolation(
                f"Tool {tool_name!r} is ADMIN level. "
                "Autonomous ADMIN actions are not permitted by policy. "
                "Escalate to a human operator via your ticketing system."
            )

        # Tools above the autonomous ceiling require human gate approval
        if spec.level > self._max_auto:
            approved = self._gate_fn(tool_name, args)
            if not approved:
                self._log(tool_name, args, "DENIED_GATE")
                raise PolicyViolation(
                    f"Tool {tool_name!r} ({spec.level.name} level) requires human "
                    "approval and the request was denied. "
                    "The requested action was not performed."
                )
            self._log(tool_name, args, "APPROVED_GATE")
        else:
            self._log(tool_name, args, "APPROVED_AUTO")

        # Execute the tool
        return spec.handler(**args)

    def get_tool_schemas(self) -> list[dict]:
        """
        Return Anthropic-format tool schemas for all non-ADMIN tools.
        ADMIN tools are never exposed to the model.
        """
        schemas = []
        for spec in self._tools.values():
            if spec.level == PermissionLevel.ADMIN:
                continue  # never expose ADMIN tools to the model
            schemas.append({
                "name": spec.name,
                "description": f"[{spec.level.name}] {spec.description}",
                "input_schema": {
                    "type": "object",
                    "properties": {},
                    "required": [],
                },
            })
        return schemas

    def audit_log(self) -> list[dict]:
        """Return a copy of the full audit log for this session."""
        return list(self._audit_log)

    def print_audit_log(self) -> None:
        """Pretty-print the audit log to stdout."""
        print("\n=== Audit Log ===")
        for entry in self._audit_log:
            status = "OK " if "APPROVED" in entry["outcome"] else "ERR"
            print(
                f"  [{status}] {entry['ts'][:19]} | "
                f"{entry['tool']:20s} | {entry['level']:8s} | {entry['outcome']}"
            )

    # ------------------------------------------------------------------
    # Private
    # ------------------------------------------------------------------

    def _log(self, tool: str, args: dict, outcome: str) -> None:
        level = self._tools[tool].level.name if tool in self._tools else "UNKNOWN"
        self._audit_log.append({
            "ts": datetime.datetime.utcnow().isoformat(),
            "tool": tool,
            "level": level,
            "args": args,
            "outcome": outcome,
        })

    @staticmethod
    def _default_gate(tool_name: str, args: dict) -> bool:
        """
        Default gate: ask the operator via stdin.

        In production, replace with:
        - Slack approval workflow
        - Web UI confirmation dialog
        - Ticketing system (PagerDuty, Linear)
        - Time-boxed async approval queue
        """
        print(f"\n[APPROVAL REQUIRED]")
        print(f"  Tool    : {tool_name}")
        print(f"  Args    : {json.dumps(args, indent=4)}")
        answer = input("  Approve? [y/N] ").strip().lower()
        return answer == "y"

### Agent loop with policy enforcement

In [ ]:
def run_agent_with_policy(
    user_task: str,
    policy: ToolPermissionPolicy,
    max_turns: int = 5,
) -> str:
    """
    A minimal agent loop that enforces the permission policy on every tool call.

    Key property: the model never executes a tool directly.
    Every call goes through policy.execute(), which checks permission level,
    fires the gate if needed, logs the outcome, and only then runs the handler.
    """
    client = anthropic.Anthropic()
    messages: list[dict] = [{"role": "user", "content": user_task}]
    tool_schemas = policy.get_tool_schemas()

    for turn in range(max_turns):
        response = client.messages.create(
            model="claude-3-5-haiku-20241022",
            max_tokens=1024,
            tools=tool_schemas,
            messages=messages,
        )

        # Model finished without requesting a tool
        if response.stop_reason == "end_turn":
            text_blocks = [b.text for b in response.content if hasattr(b, "text")]
            return "\n".join(text_blocks)

        # Collect and process tool calls
        tool_calls = [b for b in response.content if b.type == "tool_use"]
        tool_results = []

        for call in tool_calls:
            print(f"\n[Agent requesting tool] {call.name}({json.dumps(call.input)})")
            try:
                result = policy.execute(call.name, call.input)
                print(f"[Tool result] {result[:120]}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": call.id,
                    "content": result,
                })
            except PolicyViolation as e:
                # Return the denial reason to the model so it can explain to the user
                denial_msg = str(e)
                print(f"[Policy violation] {denial_msg}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": call.id,
                    "is_error": True,
                    "content": denial_msg,
                })

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

    return "Agent reached maximum turns without completing the task."

### Mock tool handlers

In [ ]:
def search_kb(query: str = "") -> str:
    return (
        f"[KB search: '{query}']\n"
        "Found 3 articles: 'Refund Policy', 'Return Window', 'Exceptions'.\n"
        "Top result: Refunds are processed within 5 business days."
    )


def send_email(to: str = "", subject: str = "", body: str = "") -> str:
    return f"[Email sent to {to!r} | subject: {subject!r}]"


def update_record(table: str = "", record_id: str = "", updates: str = "") -> str:
    return f"[Updated {table} record {record_id}: {updates}]"


def delete_records(table: str = "", condition: str = "") -> str:
    # This handler should never be called - ADMIN level blocks it
    return f"[DELETED rows from {table} WHERE {condition}]"

### Demo

In [ ]:
def demo_policy_enforcement():
    """Demonstrate all four permission outcomes without a live API call."""
    print("=" * 60)
    print("Demo: ToolPermissionPolicy enforcement (no API call)")
    print("=" * 60)

    tools = [
        ToolSpec("search_kb",     PermissionLevel.READ,    "Search knowledge base",    search_kb),
        ToolSpec("update_record", PermissionLevel.WRITE,   "Update a database record", update_record),
        ToolSpec("send_email",    PermissionLevel.EXECUTE, "Send email",               send_email),
        ToolSpec("delete_records",PermissionLevel.ADMIN,   "Delete database records",  delete_records),
    ]

    # Gate function: auto-deny for demo purposes
    policy = ToolPermissionPolicy(
        tools=tools,
        max_autonomous_level=PermissionLevel.READ,
        gate_fn=lambda tool, args: False,  # always deny in demo
    )

    # 1. READ - should pass without gate
    print("\n1. READ tool (search_kb) - expect: APPROVED_AUTO")
    result = policy.execute("search_kb", {"query": "refund policy"})
    print(f"   Result: {result[:80]}")

    # 2. WRITE - above autonomous ceiling, gate fires and denies
    print("\n2. WRITE tool (update_record) - expect: DENIED_GATE")
    try:
        policy.execute("update_record", {"table": "orders", "record_id": "42", "updates": "status=closed"})
    except PolicyViolation as e:
        print(f"   Blocked: {str(e)[:100]}")

    # 3. EXECUTE - above autonomous ceiling, gate fires and denies
    print("\n3. EXECUTE tool (send_email) - expect: DENIED_GATE")
    try:
        policy.execute("send_email", {"to": "user@example.com", "subject": "Hi", "body": "..."})
    except PolicyViolation as e:
        print(f"   Blocked: {str(e)[:100]}")

    # 4. ADMIN - unconditionally blocked, gate never fires
    print("\n4. ADMIN tool (delete_records) - expect: DENIED_ADMIN")
    try:
        policy.execute("delete_records", {"table": "users", "condition": "1=1"})
    except PolicyViolation as e:
        print(f"   Blocked: {str(e)[:100]}")

    # 5. Unknown tool
    print("\n5. Unknown tool (drop_table) - expect: DENIED_UNKNOWN")
    try:
        policy.execute("drop_table", {"table": "users"})
    except PolicyViolation as e:
        print(f"   Blocked: {str(e)[:100]}")

    policy.print_audit_log()


def demo_agent(task: str = "Search for our refund policy and summarize it in one sentence."):
    """Run the agent loop with a READ-only tool manifest."""
    print("\n" + "=" * 60)
    print("Demo: Agent loop with ToolPermissionPolicy")
    print(f"Task: {task}")
    print("=" * 60)

    tools = [
        ToolSpec("search_kb", PermissionLevel.READ, "Search the knowledge base", search_kb),
    ]
    policy = ToolPermissionPolicy(tools=tools, max_autonomous_level=PermissionLevel.READ)

    result = run_agent_with_policy(task, policy)
    print(f"\nFinal answer: {result}")
    policy.print_audit_log()

### Demo

In [ ]:
# Run policy enforcement demo (no API key needed)
demo_policy_enforcement()

# Uncomment to run the live agent demo (requires ANTHROPIC_API_KEY)
# demo_agent()

---

## Self-Check

Answer these without running code first:

**1. Which architectural flaw directly enabled this attack?**

- A. The system prompt did not explicitly prohibit deleting channels
- B. The agent had ADMIN and EXECUTE tools available with no autonomous permission ceiling
- C. The model misunderstood the user's intent
- D. The PDF parser did not strip hidden text before passing content to the agent

**2. What is the correct response to this suggestion?**

- A. Accept it - having extra tools available improves agent flexibility
- B. Accept it but set the gate function to auto-approve WRITE actions
- C. Reject it - tools not required for the declared task should not be in the manifest
- D. Accept it only if the system prompt says 'only use read tools'

**3. What should the gate function do after a timeout?**

- A. Auto-approve after timeout to avoid blocking the agent indefinitely
- B. Auto-deny and return a PolicyViolation with the timeout reason
- C. Retry the gate every 30 minutes until the operator responds
- D. Escalate to the ADMIN permission level to unblock the flow

**4. What does the absence of gate events most likely indicate?**

- A. The agent became better at solving tasks using only READ tools
- B. The WRITE tools are no longer needed for the agent's tasks
- C. A code change bypassed the policy, dispatching tools directly without going through execute()
- D. The gate function was upgraded to a faster approval system

**5. Which approach best applies the principle of least privilege?**

- A. One agent with all three tools, max_autonomous_level=ADMIN
- B. One agent with all three tools, max_autonomous_level=READ, ADMIN actions blocked unconditionally
- C. Two agents: a READ/EXECUTE agent for orders and email, plus a separate human-operated script for ADMIN cleanup
- D. One agent with max_autonomous_level=EXECUTE, ADMIN tools gated via Slack approval

**6. What is the most accurate rebuttal?**

- A. The system prompt approach is sufficient if the model has a low hallucination rate
- B. The system prompt is interpreted text that can be overridden by prompt injection; a code-level policy enforces constraints regardless of model state
- C. Code-level policies add latency; for low-risk tasks the system prompt approach is preferred
- D. The system prompt approach is fine for external-facing agents but not for internal tools

**7. What is wrong with this setup, regardless of the denial behavior?**

- A. The model should not receive PolicyViolation messages - they should be silently swallowed
- B. The ADMIN tool delete_user should never be included in get_tool_schemas(); ADMIN tools must never be exposed to the model
- C. Returning is_error=True confuses the model and degrades output quality
- D. The tool schemas should not include permission level labels in the description

**8. What does a 99.8% approval rate on gate requests indicate, and what should you do?**

- A. The gate is working perfectly - high approval means the agent is making good decisions
- B. The gate has become rubber-stamping - operators are approving without reviewing; either raise WRITE to autonomous or improve gate UX to force review
- C. The approval rate is irrelevant - what matters is that a gate exists
- D. The agent should be promoted to max_autonomous_level=WRITE since operators consistently approve

_Answers are in `checks.json` in the lesson directory._